In [0]:
# ----------------------------------------------------------------------------------------
# SCRIPT: 3_comparativo_eleitoral
# LOCAL: 3_gold/src/2_calendario_eventos/
# OBJETIVO: Comparativo de frequência antes e depois de períodos eleitorais
# ----------------------------------------------------------------------------------------


from pyspark.sql.functions import col, when, count, round

# Fato Eventos
df_fato = spark.table("gold_fato_eventos")

# Data de corte representando o inicio oficial da campanha
data_corte_eleicao = "2025-04-01"

# Classificar e Pivotar
df_resultado = df_fato.withColumn(
    "periodo",
    when(col("id_data") < data_corte_eleicao, "pre_eleicao").otherwise("periodo_eleitoral")
) \
.groupBy("descricaoTipo") \
.pivot("periodo") \
.agg(count("id_evento")) \
.withColumnRenamed("descricaoTipo", "evento") \
.fillna(0)

# Salvar na Gold
df_resultado.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_comparativo_frequencia_eleitoral")

display(df_resultado)

In [0]:
from pyspark.sql.functions import col, when, count, round

# Fato Eventos
df_fato = spark.table("gold_fato_eventos")

# Data de corte representando o inicio oficial da campanha
data_corte_eleicao = "2025-04-01"

# Classificar
df_comparativo = df_fato.withColumn(
    "periodo_eleitoral",
    when(col("id_data") < data_corte_eleicao, "Pré-Eleição").otherwise("Período Eleitoral")
)

df_resultado = df_comparativo.groupBy("descricaoTipo", "periodo_eleitoral") \
    .agg(count("id_evento").alias("total_eventos")) \
    .orderBy("descricaoTipo", "periodo_eleitoral")

# Salvar na Gold
df_resultado.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_comparativo_frequencia_eleitoral")

display(df_resultado)